<a href="https://colab.research.google.com/github/ansonkwokth/TableTennisPrediction/blob/dev/Siamese.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/ansonkwokth/TableTennisPrediction.git
%cd TableTennisPrediction

fatal: destination path 'TableTennisPrediction' already exists and is not an empty directory.
/content/TableTennisPrediction


In [2]:

import pandas as pd
from utils import data_loader as dl

import numpy as np
from model.Elo import Elo
from model.ModifiedElo import ModifiedElo
from model.ensemble import BaggingRatingSystem

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

import copy
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# Data

In [3]:
# GAME = 'TTStar'
# GAME = 'TTCup'
# GAME = 'SetkaCup'
GAME = 'SetkaCupWomen'
# GAME = 'LigaPro'


In [4]:
match GAME:
    case 'TTStar':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'TTCup':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'SetkaCup':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'SetkaCupWomen':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'LigaPro':
        years = [2022, 2023, 2024]
    case _:
        raise ValueError("Invalid game selected.")


text_data_game = dl.load_game_data(GAME, years, '../')
text_data = {
    year: text_data_game[year] for year in years
}
df = dl.create_game_dfs(GAME, years, text_data)

Loading ..//SetkaCupWomen2020.txt
Loading ..//SetkaCupWomen2021.txt
Loading ..//SetkaCupWomen2022.txt
Loading ..//SetkaCupWomen2023.txt
Loading ..//SetkaCupWomen2024.txt


In [5]:
# Generate ID indices for each pair of rows in the DataFrame
idx_lt = [i for i in range(len(df) // 2) for _ in range(2)]
df['ID'] = idx_lt  # Assign to the 'ID' column

# Reset the DataFrame index to ensure it's sequential
df.reset_index(drop=True, inplace=True)

# Get unique players and store them in player_lt
player_lt = df['Player'].unique()



In [6]:
year_val = years[-2]
year_test = years[-1]

df_train = df.loc[pd.DatetimeIndex(df['Date']).year < year_val]
df_val = df.loc[pd.DatetimeIndex(df['Date']).year == year_val]
df_test = df.loc[pd.DatetimeIndex(df['Date']).year == year_test]

In [7]:
def format_to_array(df: pd.DataFrame) -> np.ndarray:

    # info_col = ['ID', 'Round', 'Datetime', 'Game', 'Date', 'Time']
    info_col = ['Round', 'Datetime', 'Game', 'Date', 'Time']
    col = [item for item in df.columns if item not in info_col]

    df[[c for c in col if "Set" in c]] = df[[c for c in col if "Set" in c]].astype(float)
    X = df[col].values.reshape(-1, 2, len(col))
    return X

In [8]:
X_train = format_to_array(df_train)
X_val = format_to_array(df_val)
X_test = format_to_array(df_test)

In [9]:
X_all = format_to_array(df)

In [10]:
X_all

array([[[0, 'Nerush L.', 3.0, ..., 6.0, nan, nan],
        [0, 'Nerush P.', 11.0, ..., 11.0, nan, nan]],

       [[1, 'Maliuta M.', 11.0, ..., 11.0, nan, nan],
        [1, 'Kregul O.', 5.0, ..., 9.0, nan, nan]],

       [[2, 'Muliarchuk A.', 5.0, ..., nan, nan, nan],
        [2, 'Hassan A.', 11.0, ..., nan, nan, nan]],

       ...,

       [[34576, 'Lapa H.', 11.0, ..., nan, nan, nan],
        [34576, 'Man A.', 9.0, ..., nan, nan, nan]],

       [[34577, 'Lifanova O.', 11.0, ..., nan, nan, nan],
        [34577, 'Hordynska-Sheiko A.', 7.0, ..., nan, nan, nan]],

       [[34578, 'Volgina A.', 11.0, ..., 11.0, nan, nan],
        [34578, 'Gordeets M.', 6.0, ..., 5.0, nan, nan]]], dtype=object)

In [11]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class TableTennisSetScoreDataset(Dataset):
    def __init__(self, data_array, player_to_idx, score_start=2):
        """
        data_array: NumPy array of shape (n_games, 2, n_features)
          where each row is [game_index, player_name, set_score1, set_score2, ...]
        player_to_idx: dict mapping player names to unique indices.
        score_start: the column index where set scores start.
        """
        self.data_array = data_array
        self.player_to_idx = player_to_idx
        self.score_start = score_start
        # Assume that all games have the same number of score columns.
        self.max_sets = data_array.shape[2] - score_start

    def __len__(self):
        return self.data_array.shape[0]

    def __getitem__(self, idx):
        # Get the game record (2 rows, one for each player)
        game = self.data_array[idx]

        # Extract rows for player1 and player2.
        # Each row: [game_index, player_name, set_score1, set_score2, ...]
        p1_row = game[0]
        p2_row = game[1]

        # Extract player names and convert to indices.
        p1_name = p1_row[1]
        p2_name = p2_row[1]
        p1_idx = self.player_to_idx[p1_name]
        p2_idx = self.player_to_idx[p2_name]

        # Extract set scores for each player.
        p1_scores = np.array(p1_row[self.score_start:], dtype=np.float32)
        p2_scores = np.array(p2_row[self.score_start:], dtype=np.float32)

        # Create a mask for valid sets (assume both players have valid scores for the same sets).
        # True where the score is not NaN.
        mask = ~np.isnan(p1_scores)

        # Replace NaN values with 0 for numerical stability.
        p1_scores = np.nan_to_num(p1_scores, nan=0.0)
        p2_scores = np.nan_to_num(p2_scores, nan=0.0)

        # Convert everything to torch tensors.
        p1_idx = torch.tensor(p1_idx, dtype=torch.long)
        p2_idx = torch.tensor(p2_idx, dtype=torch.long)

        p1_scores = torch.tensor(p1_scores, dtype=torch.float)
        p2_scores = torch.tensor(p2_scores, dtype=torch.float)
        mask = torch.tensor(mask.astype(np.float32), dtype=torch.float)

        return p1_idx, p2_idx, p1_scores, p2_scores, mask

def create_player_mapping(data_array):
    """
    Build a dictionary mapping player names to unique indices.
    """
    players = set()
    for game in data_array:
        players.add(game[0][1])
        players.add(game[1][1])

    player_to_idx = {player: idx for idx, player in enumerate(sorted(players))}
    return player_to_idx



# Create the player mapping.
player_to_idx = create_player_mapping(X_all)

# Create the dataset and DataLoader.
dataset = TableTennisSetScoreDataset(X_all, player_to_idx)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# # Example iteration over the DataLoader.
# for batch in dataloader:
#     p1_idx, p2_idx, p1_scores, p2_scores, mask = batch
#     print("Player1 indices:", p1_idx)
#     print("Player2 indices:", p2_idx)
#     print("Player1 set scores:\n", p1_scores)
#     print("Player2 set scores:\n", p2_scores)
#     print("Mask (valid sets):\n", mask)
#     sfd
#     print()

In [38]:
import torch
import torch.nn as nn

class Siamese(nn.Module):
    def __init__(self, num_players, embedding_dim):
        """
        Args:
            num_players (int): Number of unique players.
            embedding_dim (int): Dimension of the player embedding.
        """
        super(Siamese, self).__init__()
        # Embedding layer to convert player indices to dense vectors.
        self.embedding = nn.Embedding(num_players, embedding_dim)
        # Symmetric branch: processes the sum of embeddings.
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(embedding_dim, 1),
            # nn.ReLU(),
            # nn.Linear(8, 4),
            # nn.ReLU(),
            # nn.Linear(4, 1),
        )

    def forward(self, p):
        # Look up player embeddings.
        emb = self.embedding(p)
        logits = self.linear_relu_stack(emb)
        return logits

In [39]:
def my_loss(pred1, pred2, p1_scores, p2_scores, mask):
    """
    Computes the Mean Squared Error loss only over the valid set scores.

    Args:
        pred (Tensor): Predicted scores, shape [batch, max_sets].
        target (Tensor): Ground truth scores, shape [batch, max_sets].
        mask (Tensor): Mask with 1 for valid sets and 0 for missing sets, shape [batch, max_sets].
    """
    scores_sum = p1_scores + p2_scores
    p_pred = torch.nn.Sigmoid()(-(pred1 - pred2))
    t1 = p1_scores / scores_sum
    loss1 = t1 * torch.log(p_pred)
    loss2 = (1 - t1) * torch.log(1 - p_pred)

    loss = (loss1 + loss2) * mask
    loss = torch.nan_to_num(loss, nan=0.0)


    return loss.sum()


In [40]:


# Forward pass.
# model('Slashova M.')
# model(torch.tensor(0))


In [42]:

num_players = len(player_to_idx)        # For example, 20 unique players.
embedding_dim = 16      # Embedding vector size.

# Initialize the model.
model = Siamese(num_players, embedding_dim)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 1

model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    ii = 0
    for batch in dataloader:
        # Unpack the batch.
        # p1_idx, p2_idx: player indices (shape: [batch])
        # p1_scores, p2_scores: set scores (shape: [batch, max_sets])
        # mask: binary mask for valid set scores (shape: [batch, max_sets])
        p1_idx, p2_idx, p1_scores, p2_scores, mask = batch

        optimizer.zero_grad()

        # Forward pass.
        pred_score1 = model(p1_idx)
        pred_score2 = model(p2_idx)
        print(p1_idx, p2_idx)
        print(p1_scores, p2_scores)
        print(pred_score1, pred_score2)

        # Compute losses for each player's predicted set scores.
        loss = my_loss(pred_score1, pred_score2, p1_scores, p2_scores, mask)
        print(loss)

        # Backward pass and optimization.
        loss.backward()
        optimizer.step()
        loss = my_loss(pred_score1, pred_score2, p1_scores, p2_scores, mask)
        print(loss)
        print()
        total_loss += loss.item()


        ii += 1
        if ii == 2: break


    print()
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

tensor([14]) tensor([53])
tensor([[11.,  8., 11.,  4.,  9.,  0.]]) tensor([[ 9., 11.,  9., 11., 11.,  0.]])
tensor([[1.0118]], grad_fn=<AddmmBackward0>) tensor([[0.1020]], grad_fn=<AddmmBackward0>)
tensor(-3.7275, grad_fn=<SumBackward0>)
tensor(-3.7275, grad_fn=<SumBackward0>)

tensor([54]) tensor([25])
tensor([[11., 11.,  6.,  4., 11.,  0.]]) tensor([[ 9.,  3., 11., 11.,  5.,  0.]])
tensor([[nan]], grad_fn=<AddmmBackward0>) tensor([[nan]], grad_fn=<AddmmBackward0>)
tensor(0., grad_fn=<SumBackward0>)
tensor(0., grad_fn=<SumBackward0>)


Epoch 1/1, Loss: -0.0001


In [27]:
model(torch.tensor(0))

tensor([nan], grad_fn=<ViewBackward0>)